[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C10/E10-2_Anomalies_Pompe.ipynb)

# Cycles anormaux d'une pompe : détecter sans étiquette, juger des cycles nouveaux

## Le contexte

Une pompe de procédé enchaîne des cycles de charge. Pour chacun, la supervision enregistre la durée du cycle, la pression de refoulement, le débit volumique et l'amplitude de vibration du palier. Quelques cycles se sont mal passés (fuite interne, cavitation, régime inhabituel) : la maintenance les a repérés après coup et marqués dans une colonne de référence. En exploitation, cette colonne n'existe pas à l'avance, et ces cycles passent pour ordinaires : chaque mesure, prise seule, reste dans la plage habituelle.

On veut un détecteur qui apprenne le fonctionnement habituel sur l'historique, puis qui dise, pour chaque cycle nouveau, s'il est normal ou s'il mérite une inspection. Une fausse alerte coûte une inspection inutile ; un cycle anormal laissé passer peut coûter la pompe.

C'est un problème de détection d'anomalies sans étiquette : la colonne de référence n'entre jamais dans l'apprentissage du détecteur. Sur l'historique, elle fixe le budget d'alertes et note les essais ; sur la base de test, elle note une seule fois.

Défi : déclarer normal ou anormal chacun des cycles de la base de test, et obtenir un F-score macro supérieur à celui du modèle de départ. Ce modèle de départ est une forêt d'isolement avec ses réglages par défaut, apprise sur les quatre mesures ; il atteint un F-score macro de 0,567 : il signale beaucoup de cycles ordinaires. À vous de réduire les fausses alertes sans manquer de cycles anormaux, en choisissant le détecteur et le seuil sur l'apprentissage seul ; le budget d'alertes peut se lire sur l'historique. Deux bases servent de repères, sans autre réglage que ce budget : la règle de Tukey du chapitre 3, qui ne voit aucun de ces cycles, et la distance de Mahalanobis, qui classe les cycles par leur écart à la moyenne de l'historique, rapporté à sa covariance. Un détecteur doit faire mieux que la première et se juger contre la seconde. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Anomalies_Pompe_V2.xlsx`, dans le dossier `donnees` du dépôt : 600 cycles, 5 colonnes. Une ligne est un cycle de charge.

| Colonne | Signification | Unité |
|---|---|---|
| `duree_cycle_min` | durée du cycle de charge | min |
| `pression_bar` | pression de refoulement | bar |
| `debit_m3_h` | débit volumique | m³/h |
| `vibration_um` | amplitude de vibration du palier | µm |
| `anomalie_reference` | 1 pour un cycle marqué anormal après coup, 0 sinon ; n'entre jamais dans l'apprentissage | 0 ou 1 |

Le jeu est simulé : 576 cycles normaux tirés de deux régimes de pompage, un régime partiel de cycles courts à basse pression et un régime nominal de cycles longs à haute pression ; le débit suit la racine de la pression, et la vibration croît avec la pression et la durée. S'y ajoutent 24 cycles anormaux de combinaison : débit trop faible pour la pression, vibration trop forte pour la charge, cycle court à la pression du régime nominal. Chaque valeur d'un cycle anormal reste dans la plage des cycles normaux, de sorte qu'aucune mesure ne le trahit seule. Le jeu enseigne une méthode et ne remplace pas une étude de surveillance.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et le nombre de valeurs distinctes de chaque colonne. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Anomalies_Pompe_V2.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Anomalies_Pompe_V2.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
data = pd.read_excel(dossier / FICHIER)

print("dimensions :", data.shape)
print("cases vides :", int(data.isna().sum().sum()))
uniques = data.nunique()
print("valeurs distinctes par colonne :")
print(uniques.to_string())

Lecture du résultat : 600 cycles, 5 colonnes, aucune case vide. Chaque mesure compte au moins 192 valeurs distinctes : ce sont des mesures continues, sans code ni constante à écarter. La colonne de référence n'en compte que deux, 0 et 1.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Anomalies_Pompe_V2.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : séparer l'historique et les cycles nouveaux

Un quart des cycles est mis de côté avant tout apprentissage : c'est la base de test du défi, qui joue le rôle des cycles nouveaux. La graine 42 fixe le tirage ; chacun obtient la même base de test, et les scores se comparent. Le tirage est stratifié sur la colonne de référence : les cycles anormaux étant rares, un tirage au hasard simple pourrait laisser la base de test presque sans cycle anormal, et son score ne dirait rien. Le détecteur n'apprend que sur les mesures de l'historique `X_train`. La colonne `y_train` sert à lire l'historique ; `y_test` ne sert qu'à noter, une seule fois.

In [ ]:
from sklearn.model_selection import train_test_split

mesures = [c for c in data.columns if c != "anomalie_reference"]
X, y = data[mesures], data["anomalie_reference"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=GRAINE, stratify=y)
print("historique :", len(X_train), "cycles ; base de test :", len(X_test), "cycles")
print("cycles marqués dans l'historique :", int(y_train.sum()))

### Étape 3 : regarder l'historique

Quatre vues par couples de variables, sur les cycles d'apprentissage seulement. Les cycles marqués après coup par la maintenance portent une croix.

In [ ]:
pairs = [(0, 1), (0, 2), (1, 2), (1, 3)]
marque = y_train == 1
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
for ax, (a, b) in zip(axes.ravel(), pairs):
    ax.scatter(X_train[mesures[a]], X_train[mesures[b]], s=18, alpha=0.7, color="0.35",
               label="cycle de l'historique")
    ax.scatter(X_train.loc[marque, mesures[a]], X_train.loc[marque, mesures[b]], s=45, marker="x",
               color="black", label="cycle marqué après coup")
    ax.set_xlabel(mesures[a])
    ax.set_ylabel(mesures[b])
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

Lecture du résultat : les cycles se rangent en deux nuages, des cycles courts à basse pression et des cycles longs à haute pression ; le débit suit la pression le long d'une courbe étroite, et la vibration croît avec elle. Les cycles marqués (croix) restent dans la plage de chaque mesure, mais s'écartent de ces tendances : des débits sous la courbe, des vibrations au-dessus du nuage, des cycles courts à haute pression. Un seuil sur une seule mesure ne les trouve pas ; le détecteur doit lire les mesures ensemble.

### Étape 4 : construire le modèle de départ

Le modèle de départ est volontairement simple : une forêt d'isolement avec ses réglages par défaut, sur les quatre mesures brutes. Elle apprend sur `X_train` seul. Son seuil par défaut déclare anormal tout cycle dont le score d'anomalie dépasse 0,5. La graine `random_state=0` fixe le tirage des coupures.

In [ ]:
# Votre code :

### Étape 5 : mesurer le modèle de départ

La méthode `predict` rend 1 pour un cycle jugé normal et -1 pour un cycle jugé anormal. On compte les alertes, puis on calcule la précision (part des alertes justifiées), le rappel (part des cycles anormaux retrouvés) et le F-score macro, moyenne des F-scores des deux classes. La colonne de référence de la base de test, `y_test`, sert ici pour la première et unique fois.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

pred = model.predict(X_test)                     # 1 : normal ; -1 : anormal
alerte = (pred == -1).astype(int)
print("alertes :", alerte.sum(), "sur", len(alerte), "cycles")
print(f"précision : {precision_score(y_test, alerte):.3f}")
print(f"rappel : {recall_score(y_test, alerte):.3f}")
print(f"F-score macro : {f1_score(y_test, alerte, average='macro'):.3f}")
print("matrice de confusion (lignes : réel, colonnes : prédit ; anormal, normal)")
print(confusion_matrix(y_test, alerte, labels=[1, 0]))

abscisse, ordonnee = X_test[mesures[0]], X_test[mesures[1]]
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(abscisse, ordonnee, s=18, color="0.6", label="cycle de la base de test")
ax.scatter(abscisse[alerte == 1], ordonnee[alerte == 1], s=80, marker="s", facecolor="none",
           edgecolor="black", label="alerte du modèle de départ")
ax.scatter(abscisse[y_test == 1], ordonnee[y_test == 1], s=40, marker="x", color="black",
           label="cycle marqué anormal")
ax.set_xlabel("durée du cycle (min)")
ax.set_ylabel("pression de refoulement (bar)")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.3)
plt.show()

Lecture du résultat : 34 alertes pour 150 cycles nouveaux. Le modèle retrouve cinq des six cycles marqués, avec un rappel de 0,833, mais 29 alertes sur 34 sont fausses : la précision tombe à 0,147. Le F-score macro vaut 0,567. Sur la figure, les fausses alertes tombent surtout au bord des deux régimes : le seuil par défaut, un score supérieur à 0,5, est indulgent. Avec six cycles marqués seulement parmi 150, chaque cycle compte : le F-score macro progresse par marches. C'est ce score, mesuré sur des cycles que le détecteur n'a jamais vus, qui sert de référence.

## Pour aller plus loin : moins de fausses alertes, détecteur et seuil choisis sur l'historique

Le modèle de départ reste la référence du défi. Les deux cellules suivantes montrent comment la battre sans regarder le test pour choisir. La première travaille sur l'historique seul : elle vérifie que la mise à l'échelle ne change rien à la forêt, applique la règle de Tukey du chapitre 3, base sans apprentissage à laquelle tout détecteur se compare, lit sur la colonne de référence de l'historique le budget d'alertes, puis juge à ce budget la forêt, LOF à trois voisinages et la distance de Mahalanobis, seconde base, à classement. La seconde applique le détecteur retenu parmi les deux de la fiche, LOF en mode nouveauté, aux cycles de la base de test, une seule fois. Les lectures suivent les sorties.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : sur l'historique, la mise à l'échelle laisse les scores de la forêt inchangés, et la règle de Tukey ne signale aucun des 450 cycles : les 18 cycles marqués restent dans les bornes de chaque mesure. Le budget de 18 alertes, soit 4 %, est la part des cycles marqués de l'historique. À budget égal, la forêt retrouve 6 des 18 cycles marqués, LOF, à 20 voisins, 13, et la distance de Mahalanobis 16, avec un F-score macro de 0,942 : sa covariance décrit le nuage allongé de ce jeu simulé, que les coupures de la forêt séparent probablement moins bien. De 5 à 50 voisins, le F-score macro de LOF sur l'historique reste entre 0,826 et 0,884 : avec 18 cycles marqués, il ne départage pas ces valeurs, et l'on garde la valeur par défaut, 20. Le détecteur retenu parmi les deux de la fiche, LOF à 20 voisins en mode nouveauté avec ce budget, signale 6 cycles sur 150 et atteint un F-score macro de 0,826 sur la base de test, mesuré une seule fois, contre 0,567 pour le modèle de départ. Avec six cycles marqués seulement, un cycle de plus ou de moins dans les alertes déplace le score de plusieurs centièmes.

## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 5 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = f1_score(y_test, (model.predict(X_test) == -1).astype(int), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.3f" % score)

## Synthèse

1. Sans étiquette, un détecteur apprend le fonctionnement habituel et classe les cycles du plus banal au plus singulier ; le seuil, lui, reste une décision.
2. Le seuil par défaut de la forêt d'isolement, un score supérieur à 0,5, signale beaucoup de cycles ordinaires : 34 alertes pour 6 cycles marqués sur la base de test.
3. Des cycles dont chaque mesure reste dans la plage habituelle échappent à la règle de Tukey, qui juge chaque mesure seule : il faut lire les mesures ensemble.
4. LOF juge la densité relative et demande des variables réduites ; à budget égal, il retrouve 13 des 18 cycles marqués de l'historique, contre 6 pour la forêt et 16 pour la distance de Mahalanobis, base de comparaison qu'un détecteur doit toujours affronter ; en mode nouveauté (`novelty=True`), il note des cycles qu'il n'a jamais vus sans les réapprendre.
5. Le budget d'alertes se lit sur l'historique, par la part de cycles marqués ; le test ne sert qu'à mesurer, une fois.
6. Un cycle signalé reste un cycle inhabituel, que seule une inspection qualifie.